# Cook et al. (`cook`) — sequential (Gauss-Seidel) relaxation of the three constraints (walkthrough)

**What this unrolls**, in the order the harness runs it (`evaluation.py::experiment_tracking`, `cook` branch):

| step | code |
|---|---|
| run config, frame-0 R from gyro | `run_config.py::RunConfig` |
| event frames `V` | `interacting_maps/data_loader.py::EventFrameSequence` |
| kinematic matrix `C`, normal equations `M⁻¹` | `interacting_maps/camera.py::build_kinematic_matrix`, `build_R_normal_equations`, `solve_R_lstsq` |
| network build | `run_config.py::make_network` (Cook branch) → `InteractingMaps.__init__` + seeded init |
| **one iteration**, every update/message written out | `interacting_maps/network.py::InteractingMaps.step` → `update_R_from_FC`, `update_F_from_VG`, `update_G_from_VF`, `update_G_from_I`, `update_I_from_G`, `update_F_from_RC`, clip |
| the `n_iters` loop on one frame | same, recorded per iteration |
| multi-frame loop + metrics | `evaluation.py::experiment_tracking`, `metrics.py::compute_metrics` |

Every unrolled block ends with an assertion against the real module function.
Inputs: `data/poster_rotation/` (seg_C — the same segment as `cmax_v1/v2_walkthrough.ipynb`, so the four notebooks
compare directly). Outputs: figures + one tracking run under `experiments/model_walkthrough/`.

**Cook = sequential updates.** Every update rule overwrites its map immediately, so the next rule in the same iteration already sees the new value (Gauss-Seidel). `R` has no anchor: it only ever moves toward the flow-implied `R* = M⁻¹ΣCᵀF`, so the angular velocity is whatever `F` encodes — and `F` is shaped by `V+F·G=0`, which is invariant to `F→βF, G→G/β` (the β-scale ambiguity, architecture.md).

In [ ]:
# setup — run from the repo root (data/ and experiments/ are relative paths)
import os, sys
os.environ.setdefault('IM_MLFLOW', '0')          # walkthrough runs never go to MLflow
ROOT = os.path.abspath('.')
assert os.path.exists(os.path.join(ROOT, 'config.py')), 'start the notebook from the repo root'
sys.path.insert(0, ROOT)
# local slice switches (env): WT_NFRAMES (default 5), WT_NITERS (default ITERS_PER_FRAME), WT_K (frame studied in detail)

In [ ]:
# imports — only what the cells use, grouped by module
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from config import ITERS_PER_FRAME
from run_config import RunConfig, make_network
from interacting_maps.data_loader import EventFrameSequence, CameraCalibration
from interacting_maps.eval_io import (load_imu, load_groundtruth, load_omega_gt,
                                      get_gyro_for_frame, get_reference_omega)
from interacting_maps.metrics import compute_metrics, curl_share
from interacting_maps.network import InteractingMaps
from interacting_maps.camera import (build_kinematic_matrix, build_R_normal_equations, solve_R_lstsq,
                                     CLIP_I, CLIP_G, CLIP_F)
from interacting_maps.style import use_style, save, to_deg_s, OMEGA_AXIS_COLORS, SCREEN_WIDTH
from interacting_maps.viz import draw_map, map_scale, net_maps
import evaluation as E   # only for the final end-to-end check (experiment_tracking)

In [ ]:
# constants — the walkthrough's knobs (same segment as the CMax walkthroughs: poster seg_C, ω_x-dominant, |ω|≈1.1 rad/s)
DATASET    = 'poster_rotation'
SEGMENT_ID = 'seg_C'                                         # config.DATASET_SEGMENTS
MODEL      = 'cook'
N_FRAMES   = int(os.environ.get('WT_NFRAMES', 5))            # segment default is 150
N_ITERS    = int(os.environ.get('WT_NITERS', ITERS_PER_FRAME))   # config.ITERS_PER_FRAME = 75
K          = int(os.environ.get('WT_K', 0))                  # frame studied step by step
OUT_ROOT   = 'experiments/model_walkthrough'                 # NOT results/ (load-bearing)
FIG_DIR    = os.path.join(OUT_ROOT, 'cook_figures')
os.makedirs(FIG_DIR, exist_ok=True)
np.set_printoptions(precision=4, suppress=True)
assert 0 <= K < N_FRAMES

## 1) Run configuration — `run_config.py::RunConfig`
`RunConfig(model='cook')` picks `COOK_PARAMS` and sets `use_thesis=False`, `use_imu=False`. `poisson` is irrelevant (Cook has its own I update) and is left out of the folder name. **`initial_R` is still the gyro** mean over the first frame window × `dt` (`RunConfig._compute_initial_R`): the frame-0 state is IMU-seeded even for this "vision-only" model.

In [ ]:
# run_config.py::RunConfig.__init__
rc = RunConfig(dataset=DATASET, model=MODEL, segment=SEGMENT_ID,
               n_frames=N_FRAMES, n_iters=N_ITERS, out_root=OUT_ROOT)
rc

In [ ]:
# the switches this model sets, the deltas, and the frame-0 R (rad/frame and °/s)
print({k: getattr(rc, k) for k in ('use_thesis', 'use_imu', 'use_cmax', 'use_cmax_v2',
                                   'distortion_mode', 'poisson', 'frame_duration')})
print('params     :', rc.params)
print('initial_R  :', rc.initial_R, 'rad/frame  =', to_deg_s(rc.initial_R / rc.frame_duration), '°/s')
print('output_dir :', rc.output_dir)

## 2) Event frames `V` — `interacting_maps/data_loader.py::EventFrameSequence`
The network's only per-frame input. Events are read once for `[t_start, t_start + n·dt + 0.1 s)`; each frame bins its
window into a signed count image, clipped at ±10 and divided by 10 (`clip_value=10.0` in `experiment_tracking`).
In `C_full` mode the events are **not** undistorted here (lens handled inside the kinematic matrix `C`).

In [ ]:
# evaluation.py::experiment_tracking — EventFrameSequence(...) exactly as the harness builds it
seq = EventFrameSequence(
    rc.paths['events'], rc.paths['calib'],
    frame_duration=rc.frame_duration, t_start=rc.t_start,
    n_frames=rc.n_frames, clip_value=10.0,
    undistort=rc.undistort_at_event_level,
    sensor_size=rc.sensor_size,
)
H, W = seq.H, seq.W
fx, fy, cx, cy = seq.calib.fx, seq.calib.fy, seq.calib.cx, seq.calib.cy
frames = list(seq)                         # [(V, t_mid), ...]
print('H, W:', H, W, '| undistort at event level:', seq.undistort, '| dist coeffs:', seq.calib.dist)

In [ ]:
# what V looks like: per frame, share of pixels with events and of saturated pixels (|V| = 1 ↔ ≥10 net events)
pd.DataFrame([{'frame': k, 't_mid': t, 'nonzero_px': int((V != 0).sum()),
               'nonzero_share': (V != 0).mean(), 'saturated_px': int((np.abs(V) >= 1).sum()),
               'V_min': V.min(), 'V_max': V.max()}
              for k, (V, t) in enumerate(frames)])

In [ ]:
# sensors: Vicon (independent SCORING reference) and the gyro (frame-0 init; for thesis_imu also the anchor)
imu_data = load_imu(rc.paths['imu'])
gt_data  = load_groundtruth(rc.paths['groundtruth'])
om_data  = load_omega_gt(rc.paths.get('omega_gt'))
print('imu rows', len(imu_data), '| gt rows', len(gt_data), '| omega_gt', om_data is not None)

## 3) Kinematic matrix `C` — `interacting_maps/camera.py::build_kinematic_matrix`
Built once in `InteractingMaps.__init__`. Per pixel a 2×3 matrix maps ω (rad/frame) to pixel flow `F = C·R` (px/frame). In `C_full`
every native (distorted) pixel is back-projected to its true undistorted normalized coords `(x', y')` with
`cv2.undistortPoints`, the rotational flow `A(x',y')` is evaluated there and mapped back into distorted-pixel space by
the Brown–Conrady Jacobian: `C = diag(fx,fy)·J_D·A`. This is the only place the lens enters the network.

In [ ]:
# run_config.py::make_network — dist_coeffs for C_full (None for undistort_events → pinhole C)
dist_coeffs = CameraCalibration(rc.paths['calib']).dist if rc.distortion_mode == 'C_full' else None
dist_coeffs

In [ ]:
# camera.py::build_kinematic_matrix — distortion-aware C, and the pinhole C for comparison
C_mat = build_kinematic_matrix(H, W, fx, fy, cx, cy, dist_coeffs=dist_coeffs, include_jacobian=True)
C_pin = build_kinematic_matrix(H, W, fx, fy, cx, cy, dist_coeffs=None)
print('C_mat', C_mat.shape)
for name, (r, c) in (('centre', (H // 2, W // 2)), ('corner', (0, 0))):
    print(f'{name} pixel (row {r}, col {c}):\n', C_mat[r, c], '\n  pinhole:\n', C_pin[r, c])

In [ ]:
# what the lens does to C: relative change of each pixel's C vs pinhole
rel = np.linalg.norm(C_mat - C_pin, axis=(2, 3)) / np.linalg.norm(C_pin, axis=(2, 3))
print(f'|C_full − C_pinhole| / |C_pinhole|: centre {rel[H // 2, W // 2]:.3%}, median {np.median(rel):.2%}, max {rel.max():.2%}')

### 3b) Normal equations — `camera.py::build_R_normal_equations`, `solve_R_lstsq`
`R* = argmin_R Σ‖F − C·R‖² = M⁻¹·ΣCᵀF` with the constant `M = ΣCᵀC` (3×3), precomputed once. Both networks move `R`
toward this `R*`. Its conditioning says how well the whole flow field pins down each axis: ω_x/ω_y produce
near-uniform translations of the image (huge weight in `M`), ω_z a rotation about the centre that is small near it.

In [ ]:
# camera.py::build_R_normal_equations — M = Σ CᵀC, inverted
M = np.einsum('hwji,hwjk->ik', C_mat, C_mat)
M_inv = np.linalg.inv(M)
print('M =\n', M, '\ncond(M) =', np.linalg.cond(M))
print('per-axis sensitivity sqrt(diag M) (px per rad/frame, summed over pixels):', np.sqrt(np.diag(M)))
assert np.allclose(M_inv, build_R_normal_equations(C_mat))

In [ ]:
# camera.py::solve_R_lstsq — R* from a flow field; on the exact field F = C·R it returns R (sanity of the solve)
F_init = np.einsum('hwij,j->hwi', C_mat, rc.initial_R)
R_star_init = M_inv @ np.einsum('hwji,hwj->i', C_mat, F_init)
assert np.allclose(R_star_init, solve_R_lstsq(M_inv, C_mat, F_init))
print('initial R  :', rc.initial_R, '\nR*(C·R)    :', R_star_init)
mag = np.hypot(F_init[..., 0], F_init[..., 1])
print(f'|F| of the initial flow C·R: mean {mag.mean():.3f} px/frame, max {mag.max():.3f}')

In [ ]:
# the initial flow field C·R_init (direction = colour, |F| = brightness)
with use_style('screen'):
    fig, ax = plt.subplots(1, 1, figsize=(SCREEN_WIDTH / 3, 2.4))
    draw_map(ax, 'F', F_init, title=r'$F = C\cdot R_{init}$')
    path = os.path.join(FIG_DIR, f'{MODEL}_F_init.png'); save(fig, path)
display(Image(path))

## 4) Network — `run_config.py::make_network` (Cook branch, unrolled)
`InteractingMaps(...)` builds `C` (as above) and `M⁻¹`. The maps are then set **by `make_network`**, not by the class:
`I` = 0.001·N(0,1) with `default_rng(42)` on an `(H+1)×(W+1)` grid (one extra row and column so forward differences
give a full `H×W` gradient), `G = 0`, `F = C·R_init`, `R = R_init`. Note the I noise is 10× smaller than the thesis
init (0.01).

In [ ]:
# run_config.py::make_network — InteractingMaps(...)
net = InteractingMaps(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy, dist_coeffs=dist_coeffs, **rc.params)
print({k: getattr(net, k) for k in ('delta_VFG', 'delta_IG', 'delta_GI', 'delta_RF', 'delta_FR',
                                    'delta_map', 'delta_shrinkI', 'delta_curl')})
assert np.array_equal(net._C_mat, C_mat) and np.allclose(net._M_inv, M_inv)

In [ ]:
# run_config.py::make_network — seeded, mutually consistent start
net.I = np.random.default_rng(42).standard_normal((H + 1, W + 1)) * 0.001
net.G = np.zeros((H, W, 2), dtype=np.float64)
net.F = np.einsum('hwij,j->hwi', net._C_mat, rc.initial_R)
net.R = rc.initial_R.copy()
print('I', net.I.shape, 'std', net.I.std(), '| G', net.G.shape, '| F', net.F.shape, '| R', net.R)

In [ ]:
# check: make_network == unrolled build
net_chk = make_network(rc, H, W, fx, fy, cx, cy)
for m in ('I', 'G', 'F', 'R'):
    assert np.array_equal(getattr(net, m), getattr(net_chk, m)), m
print('[ok] make_network == unrolled build')

## 5) Frame `K` — `evaluation.py::experiment_tracking` (per-frame inputs)
Below one frame `K` (default 0) is followed iteration by iteration. For `K > 0` the network first processes frames
`0..K−1` exactly as the harness does, so frame `K` starts from the carried-over (warm-started) maps.
The gyro is read only for comparison — Cook's `step` takes no ω. The reference `ω_ref` (Vicon, differenced) is used only to score.

In [ ]:
# evaluation.py::experiment_tracking — frames before K (warm start), then frame K's inputs
for k_prev, (V_k, _) in enumerate(frames[:K]):
    omega_imu_k = get_gyro_for_frame(imu_data, rc.t_start + k_prev * rc.frame_duration,
                                     rc.t_start + (k_prev + 1) * rc.frame_duration)
    net.step(V_k, n_iters=rc.n_iters)
t_lo = rc.t_start + K * rc.frame_duration
t_hi = t_lo + rc.frame_duration
V, t_mid = frames[K]
omega_imu = get_gyro_for_frame(imu_data, t_lo, t_hi)
omega_ref, ref_src = get_reference_omega(gt_data, imu_data, t_lo, t_hi, om_data)
print(f'frame {K}: [{t_lo:.4f}, {t_hi:.4f})  t_mid {t_mid:.4f}')
print('ω gyro  °/s :', to_deg_s(omega_imu))
print(f'ω ref   °/s : {to_deg_s(omega_ref)}   ({ref_src})')
print('network R at frame start °/s:', to_deg_s(net.R / rc.frame_duration))

## 6) One iteration — `InteractingMaps.step` body, unrolled (`n_iters = 1`)
`step` runs six updates **in this order**, each writing its map in place (Gauss-Seidel), then the regularisers
(off: `delta_map = delta_curl = 0`) and the clip. After each update a copy `net_gs` runs the real method and the maps
must be identical. `net0` keeps the frame-start state for the `step()` checks.

In [ ]:
# keep the frame-start state; net_gs follows along with the real methods
net0   = copy.deepcopy(net)
net_gs = copy.deepcopy(net)

### 6a) R from F — `update_R_from_FC`
First in the cycle: `R* = M⁻¹ΣCᵀF` from the current flow, then `R ← (1−δ_FR)·R + δ_FR·R*`. On frame 0 `F = C·R`
exactly, so `R* = R` and this first update is a no-op. **Cook does not clip R.**

In [ ]:
# network.py::update_R_from_FC — closed-form R*, then blend
R_new = solve_R_lstsq(net._M_inv, net._C_mat, net.F)
R_before = net.R.copy()
net.R = (1.0 - net.delta_FR) * net.R + net.delta_FR * R_new
print('R*  °/s:', to_deg_s(R_new / rc.frame_duration))
print('R   °/s:', to_deg_s(R_before / rc.frame_duration), '→', to_deg_s(net.R / rc.frame_duration))

In [ ]:
# check: InteractingMaps.update_R_from_FC == unrolled
net_gs.update_R_from_FC()
assert np.array_equal(net_gs.R, net.R), 'R'
print('[ok] update_R_from_FC')

### 6b) F from V, G — `update_F_from_VG`
Gradient step on `(V + F·G)²`: `e = V + F·G`, `F ← F − δ_VFG·2·G·e`. With `G = 0` (frame 0, first iteration) the step
is zero: F cannot move until G exists.

In [ ]:
# network.py::update_F_from_VG — OFCE residual, step on F
e = V + np.einsum('hwk,hwk->hw', net.F, net.G)
dF = net.delta_VFG * 2.0 * net.G * e[..., np.newaxis]
net.F -= dF
print(f'residual |e|: mean {np.abs(e).mean():.4f}, max {np.abs(e).max():.4f} | |ΔF| max {np.abs(dF).max():.4g}')

In [ ]:
# check: InteractingMaps.update_F_from_VG == unrolled
net_gs.update_F_from_VG(V)
assert np.array_equal(net_gs.F, net.F), 'F'
print('[ok] update_F_from_VG')

### 6c) G from V, F — `update_G_from_VF`
Same residual, **recomputed with the F just written** (Gauss-Seidel), step on G: `G ← G − δ_VFG·2·F·e`. On the first
iteration `G = 0 ⇒ e = V`, so `G = −2δ_VFG·V·F`: G is born parallel to the flow, with sign and size set by the events.
Steps are **not clipped** (unlike the thesis' `max_grad`).

In [ ]:
# network.py::update_G_from_VF — residual with the NEW F, step on G
e = V + np.einsum('hwk,hwk->hw', net.F, net.G)
dG = net.delta_VFG * 2.0 * net.F * e[..., np.newaxis]
net.G -= dG
print(f'|ΔG| max {np.abs(dG).max():.4f} | pixels with G ≠ 0: {(np.abs(net.G).sum(-1) > 0).mean():.2%}')

In [ ]:
# check: InteractingMaps.update_G_from_VF == unrolled
net_gs.update_G_from_VF(V)
assert np.array_equal(net_gs.G, net.G), 'G'
print('[ok] update_G_from_VF')

### 6d) G from I — `update_G_from_I`
`∇I` by forward differences on the `(H+1)×(W+1)` grid (`G[...,0] = ∂I/∂x` along columns, `G[...,1] = ∂I/∂y` along
rows), then the blend `G ← (1−δ_IG)·G + δ_IG·∇I`.

In [ ]:
# network.py::_grad_I — forward differences, full H×W thanks to the extra row/column of I
dIx = net.I[:H, 1:W + 1] - net.I[:H, :W]
dIy = net.I[1:H + 1, :W] - net.I[:H, :W]
grad_I = np.stack([dIx, dIy], axis=-1)
assert np.array_equal(grad_I, net._grad_I())
print('|∇I| mean', np.hypot(dIx, dIy).mean())

In [ ]:
# network.py::update_G_from_I — blend toward ∇I
net.G = (1.0 - net.delta_IG) * net.G + net.delta_IG * grad_I

In [ ]:
# check: InteractingMaps.update_G_from_I == unrolled
net_gs.update_G_from_I()
assert np.array_equal(net_gs.G, net.G), 'G'
print('[ok] update_G_from_I')

### 6e) I from G — `update_I_from_G`
`Ψ = G − ∇I` (∇I recomputed — I has not changed, but G has), backward differences `Ψ̂` (border = 0), and
`I[:H,:W] ← I[:H,:W] − δ_GI·(Ψ̂_x + Ψ̂_y)`: one gradient step on `‖∇I − G‖²`. **Only `I[:H,:W]` is written** — the
extra last row and column of I keep their initial noise forever, yet `_grad_I` reads them for the last column/row of
`∇I`.

In [ ]:
# network.py::update_I_from_G — residual and its backward differences
Psi = net.G - net._grad_I()
Psi_x, Psi_y = Psi[..., 0], Psi[..., 1]
Psi_hat_x = np.zeros((H, W)); Psi_hat_x[:, 0] = Psi_x[:, 0]; Psi_hat_x[:, 1:] = Psi_x[:, 1:] - Psi_x[:, :-1]
Psi_hat_y = np.zeros((H, W)); Psi_hat_y[0, :] = Psi_y[0, :]; Psi_hat_y[1:, :] = Psi_y[1:, :] - Psi_y[:-1, :]
step_I = Psi_hat_x + Psi_hat_y
print('|Ψ| mean', np.abs(Psi).mean(), '| |Ψ̂x+Ψ̂y| max', np.abs(step_I).max(), '| delta_shrinkI', net.delta_shrinkI)

In [ ]:
# network.py::update_I_from_G — the step (delta_shrinkI = 0 → no shrink term)
if net.delta_shrinkI > 0.0:
    step_I = step_I + net.delta_shrinkI * net.I[:H, :W]
I_pad_before = (net.I[H, :].copy(), net.I[:, W].copy())
net.I[:H, :W] -= net.delta_GI * step_I
print('extra row/col of I unchanged:', np.array_equal(net.I[H, :], I_pad_before[0]) and np.array_equal(net.I[:, W], I_pad_before[1]))

In [ ]:
# check: InteractingMaps.update_I_from_G == unrolled
net_gs.update_I_from_G()
assert np.array_equal(net_gs.I, net.I), 'I'
print('[ok] update_I_from_G')

### 6f) F from R — `update_F_from_RC`
Last: `F ← (1−δ_RF)·F + δ_RF·C·R` with the R from 6a. With `δ_RF = 0.03` the kinematic prior is weak: 97 % of F is
kept from the OFCE step each iteration.

In [ ]:
# network.py::update_F_from_RC — blend toward C·R
F_candidate = np.einsum('hwij,j->hwi', net._C_mat, net.R)
net.F = (1.0 - net.delta_RF) * net.F + net.delta_RF * F_candidate
print('|F − C·R| mean', np.abs(net.F - F_candidate).mean())

In [ ]:
# check: InteractingMaps.update_F_from_RC == unrolled
net_gs.update_F_from_RC()
assert np.array_equal(net_gs.F, net.F), 'F'
print('[ok] update_F_from_RC')

### 6g) Regularisers and clip
`delta_map`, `delta_curl` are 0 (Cook as published) → skipped. Then F, G, I[:H,:W] are clipped to ±10, ±5, ±10.
R is not clipped.

In [ ]:
# network.py::step — regularisers (off) and value bounds
print('delta_map', net.delta_map, '| delta_curl', net.delta_curl)
if net.delta_map > 0.0:
    net.G *= (1.0 - 2.0 * net.delta_map)
assert net.delta_curl == 0.0
np.clip(net.F, -CLIP_F, CLIP_F, out=net.F)
np.clip(net.G, -CLIP_G, CLIP_G, out=net.G)
np.clip(net.I[:H, :W], -CLIP_I, CLIP_I, out=net.I[:H, :W])
print('share at bound: F', np.mean(np.abs(net.F) >= CLIP_F), 'G', np.mean(np.abs(net.G) >= CLIP_G))

In [ ]:
# check: InteractingMaps.step(V, n_iters=1) == the unrolled iteration
net_ref = copy.deepcopy(net0)
net_ref.step(V, n_iters=1)
for m in ('I', 'G', 'F', 'R'):
    assert np.array_equal(getattr(net_ref, m), getattr(net, m)), m
print('[ok] step(n_iters=1) == unrolled iteration')

## 7) The `n_iters` loop on frame `K` — `InteractingMaps.step` (unrolled with the update methods)
Restart from `net0` and run all iterations, calling the six update methods in `step`'s order (each proven above) and
recording, per iteration, `R`, the `R*` that `update_R_from_FC` blends toward, and the two constraint residuals.
Since R has no other input, it can only follow R* — the question is whether R* (the flow) tracks the reference.

In [ ]:
# network.py::step — n_iters loop with recording
net = copy.deepcopy(net0)
R_iters, R_star, res_vfg, res_gi = [net.R.copy()], [], [], []
res_fcr, R_star_post = [], []   # kinematic residual and R* after each iteration (for the CSV)
for _ in range(rc.n_iters):
    R_star.append(solve_R_lstsq(net._M_inv, net._C_mat, net.F))
    net.update_R_from_FC()
    net.update_F_from_VG(V)
    net.update_G_from_VF(V)
    net.update_G_from_I()
    net.update_I_from_G()
    net.update_F_from_RC()
    np.clip(net.F, -CLIP_F, CLIP_F, out=net.F)
    np.clip(net.G, -CLIP_G, CLIP_G, out=net.G)
    np.clip(net.I[:H, :W], -CLIP_I, CLIP_I, out=net.I[:H, :W])
    R_iters.append(net.R.copy()); res_vfg.append(net.residual_VFG(V)); res_gi.append(net.residual_GI())
    res_fcr.append(net.residual_FCR()); R_star_post.append(solve_R_lstsq(net._M_inv, net._C_mat, net.F))
R_iters, R_star = np.array(R_iters), np.array(R_star)
print('R after frame °/s:', to_deg_s(net.R / rc.frame_duration), '| R* last iter:', to_deg_s(R_star[-1] / rc.frame_duration))

In [ ]:
# check: InteractingMaps.step == unrolled loop
net_ref = copy.deepcopy(net0)
net_ref.step(V, n_iters=rc.n_iters)
for m in ('I', 'G', 'F', 'R'):
    assert np.array_equal(getattr(net_ref, m), getattr(net, m)), m
print('[ok] step() == unrolled loop')

In [ ]:
# R over the iterations vs the flow-implied R*, the reference and the gyro (°/s)
it = np.arange(rc.n_iters + 1)
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3), sharex=True)
    for a, ax in enumerate(axes):
        ax.plot(it, to_deg_s(R_iters[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], label='network R')
        ax.plot(it[:-1], to_deg_s(R_star[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], ls='--', lw=0.8, label='R* (flow)')
        ax.axhline(to_deg_s(omega_ref[a]), color='k', lw=1.2, label='reference')
        ax.axhline(to_deg_s(omega_imu[a]), color='0.55', ls=':', label='gyro (not used)')
        ax.set_title(f'$\\omega_{"xyz"[a]}$ (°/s)'); ax.set_xlabel('iteration')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, f'{MODEL}_R_over_iterations.png'); save(fig, path)
display(Image(path))

In [ ]:
# constraint residuals over the iterations: mean |V + F·G| and mean |G − ∇I|
with use_style('screen'):
    fig, axes = plt.subplots(1, 2, figsize=(SCREEN_WIDTH * 2 / 3, 2.6), sharex=True)
    axes[0].plot(it[1:], res_vfg, color='0.25'); axes[0].set_title(r'mean $|V + F\cdot G|$')
    axes[1].plot(it[1:], res_gi, color='0.25'); axes[1].set_title(r'mean $|G - \nabla I|$')
    for ax in axes:
        ax.set_xlabel('iteration')
    path = os.path.join(FIG_DIR, f'{MODEL}_residuals.png'); save(fig, path)
display(Image(path))

In [ ]:
# save every residual per iteration of frame K (state after each iteration), for combined figures later
# R, R*, ω in °/s; res_RRstar = |R − R*| (norm, °/s)
deg = lambda r: to_deg_s(np.asarray(r) / rc.frame_duration)
it_df = pd.DataFrame({'iteration': it[1:],
                      **{f'R_w{c}': deg(R_iters[1:, a]) for a, c in enumerate('xyz')},
                      **{f'Rstar_w{c}': deg(np.array(R_star_post)[:, a]) for a, c in enumerate('xyz')},
                      'res_VFG': res_vfg, 'res_GI': res_gi, 'res_FCR': res_fcr,
                      'res_RRstar': np.linalg.norm(deg(R_iters[1:]) - deg(R_star_post), axis=1),

                      })
csv_path = os.path.join(FIG_DIR, f'{MODEL}_iterations.csv'); it_df.to_csv(csv_path, index=False)
print(csv_path, it_df.shape); it_df.iloc[[0, 1, -1]].round(4)

## 8) The maps after frame `K` — what the network inferred
`V` and the three inferred maps (cropped to `H×W` by `viz.net_maps`). Check whether `G` is concentrated on the event
pixels, whether `F` still looks like a rigid rotation field (`C·R`) or has picked up local structure from OFCE, and how
much of G is not any image's gradient (`metrics.curl_share` — the part the I recovery must discard).

In [ ]:
# viz.net_maps + draw_map — V, I, G, F after frame K
maps = net_maps(net, H, W)
with use_style('screen'):
    fig, axes = plt.subplots(1, 4, figsize=(SCREEN_WIDTH, 2.6))
    draw_map(axes[0], 'V', V, title=f'V (frame {K})')
    draw_map(axes[1], 'I', maps['I'])
    draw_map(axes[2], 'G', maps['G'])
    draw_map(axes[3], 'F', maps['F'])
    path = os.path.join(FIG_DIR, f'{MODEL}_maps_frame{K}.png'); save(fig, path)
display(Image(path))

In [ ]:
# numbers behind the picture: residuals, curl share, clip hits, F vs the rigid field C·R
F_rigid = np.einsum('hwij,j->hwi', C_mat, net.R)
ev = V != 0
print(f'mean |V+F·G| {net.residual_VFG(V):.4f} | mean |G−∇I| {net.residual_GI():.4f} | curl share of G {curl_share(maps["G"]):.2%}')
print(f'|G| mean on event px {np.hypot(*maps["G"][ev].T).mean():.4f} vs off {np.hypot(*maps["G"][~ev].T).mean():.4f}')
print(f'|F − C·R| / |C·R| = {np.linalg.norm(maps["F"] - F_rigid) / np.linalg.norm(F_rigid):.2%}')
print('share at clip bound: F', np.mean(np.abs(maps['F']) >= CLIP_F), 'G', np.mean(np.abs(maps['G']) >= CLIP_G),
      'I', np.mean(np.abs(maps['I']) >= CLIP_I), '| I std', maps['I'].std())
e, d, b = compute_metrics(net.R / rc.frame_duration, omega_ref)
print(f'frame {K}: err {e:.2f} °/s, dir {d:.2f}°, β {b:.3f}')

## 9) All frames — `evaluation.py::experiment_tracking` (`cook` branch, unrolled)
Fresh network, then per frame: gyro + reference for the window → else: `net.step(V)` → `ω_est = R/dt` → `compute_metrics`
against the reference. The maps are carried from frame to frame (warm start); `R` is never reset.

In [ ]:
# evaluation.py::experiment_tracking — the frame loop
net = make_network(rc, H, W, fx, fy, cx, cy)
rows = []
for k, (V_k, t_mid_k) in enumerate(frames):
    t_lo_k = rc.t_start + k * rc.frame_duration
    t_hi_k = t_lo_k + rc.frame_duration
    omega_imu_k = get_gyro_for_frame(imu_data, t_lo_k, t_hi_k)
    omega_ref_k, _ = get_reference_omega(gt_data, imu_data, t_lo_k, t_hi_k, om_data)
    net.step(V_k, n_iters=rc.n_iters)
    omega_est = net.R / rc.frame_duration
    err, dir_err, beta = compute_metrics(omega_est, omega_ref_k)
    rows.append({'frame': k, 't_mid': t_mid_k,
                 **{f'est_w{c}': v for c, v in zip('xyz', to_deg_s(omega_est))},
                 **{f'ref_w{c}': v for c, v in zip('xyz', to_deg_s(omega_ref_k))},
                 **{f'imu_w{c}': v for c, v in zip('xyz', to_deg_s(omega_imu_k))},
                 'err_est': err, 'err_gyro': compute_metrics(omega_imu_k, omega_ref_k)[0],
                 'dir_err': dir_err, 'beta': beta,
                 'res_VFG': net.residual_VFG(V_k), 'res_GI': net.residual_GI(), 'res_FCR': net.residual_FCR(),
                 'I_std': float(net.I[:H, :W].std()), 'curl_share': curl_share(net.G[:H, :W])})
df = pd.DataFrame(rows)
df.round(3)

In [ ]:
# summary (°/s): network vs the gyro, both against the reference
df[['err_est', 'err_gyro', 'dir_err', 'beta', 'res_VFG', 'res_GI', 'res_FCR', 'curl_share']].describe().round(3)

In [ ]:
# save the per-frame table (ω in °/s, residuals after each frame), for combined figures later
csv_path = os.path.join(FIG_DIR, f'{MODEL}_frames.csv'); df.to_csv(csv_path, index=False)
print(csv_path, df.shape)

In [ ]:
# check: evaluation.experiment_tracking (cook) == unrolled loop  (writes to experiments/model_walkthrough/)
E.experiment_tracking(rc, save_frames=False)
trk = pd.read_csv(os.path.join(rc.output_dir, 'tracking.csv'))
assert np.allclose(np.degrees(trk[['est_wx', 'est_wy', 'est_wz']].values),
                   df[['est_wx', 'est_wy', 'est_wz']].values, atol=1e-9)
assert np.allclose(trk['err_deg_s'].values, df['err_est'].values, atol=1e-9)
assert np.allclose(trk['I_std'].values, df['I_std'].values, atol=1e-12)
print('[ok] experiment_tracking == unrolled cook loop')

In [ ]:
# ω per axis over the frames: estimate vs reference vs gyro (°/s)
t_axis = df['t_mid'] - rc.t_start
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3.2), sharex=True)
    for a, ax in enumerate(axes):
        c = 'xyz'[a]
        ax.plot(t_axis, df[f'est_w{c}'], color=OMEGA_AXIS_COLORS[a], label='network ω')
        ax.plot(t_axis, df[f'ref_w{c}'], color='k', lw=1.2, label='reference')
        ax.plot(t_axis, df[f'imu_w{c}'], color='0.55', ls=':', label='gyro (not used)')
        ax.set_title(f'$\\omega_{c}$ (°/s)')
        ax.set_xlabel('time (s)')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, f'{MODEL}_omega_over_frames.png')
    save(fig, path)
display(Image(path))

## What to check
- **1 init:** the "vision-only" Cook run starts from the gyro (`initial_R = ω_gyro·dt`, `F = C·R`). How much of its
  early accuracy is that seed? (Compare frame 0 `err_est` with later frames.)
- **6a/6b first iteration:** `R* = R` and the F step is zero on frame 0 — the first cycle only creates G (6c).
- **6e frozen border:** `update_I_from_G` never writes the extra row/column of the `(H+1)×(W+1)` I, but `_grad_I` reads
  them — the last column/row of ∇I is pinned to the init noise for the whole run.
- **6c no step clip:** unlike the thesis OFCE (`max_grad = 5`), Cook's F/G steps are unbounded; only the value clip
  (F ±10, G ±5) catches a blow-up — check the clip-hit shares in 8.
- **7 R follows R*:** Cook has no anchor; if R* drifts from the reference within a frame, nothing pulls it back
  (β-scale ambiguity: `V + F·G` is invariant to `F→βF, G→G/β`). Watch `beta` in 9.
- **Gauss-Seidel vs Jacobi:** the thesis notebook runs the same three constraints simultaneously; with
  `delta_FR = 0.10` matched, the remaining differences are the update order, the OFCE step clip, the border
  convention of ∇I and the I-init scale (0.001 vs 0.01).